# Conformance checking on the Road Traffic Fine Management log

Runs the thesis conformance stack (`conformance/`, `derive/statistical_analysis.py`) on a
public, non-clinical event log to show the method is domain-agnostic.

Pipeline in this notebook:

1. **Load** `Road_Traffic_Fine_Management_Process.xes` (150,370 cases / 561,470 events) and cache it as parquet.
2. **Preprocess** an episode outcome: *good* = the full amount was paid in time, *bad* otherwise.
3. **Segment** each case into episodes triggered by `Create Fine`.
4. **Check** the Declare model in `evaluation/rtfm.yaml` per episode.
5. **Analyse** rule&nbsp;vs.&nbsp;outcome associations with the existing `run_analysis` (Fisher exact, OR, RR, BH correction).

Two conventions carried over from the SBT pipeline, both explained in `evaluation/rtfm_utils.py`:

* The log is kept **wide** (no `itemid` column) so the checker exposes each event's own attributes
  as `items['amount']`, `items['expense']`, ... Do not call `build_event_log()` on it.
* The results table reuses the pipeline's column names (`stay_id`, `sbt_outcome`, ...) verbatim, so
  `derive/statistical_analysis.py` runs unchanged. Here they mean fine/case, not ICU stay.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "evaluation" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from tqdm.auto import tqdm

from conformance.checker import evaluate_constraint
from conformance.rules import load_model_from_yaml
from evaluation.rtfm_utils import (
    build_trigger_episodes,
    derive_episode_outcome,
    load_severity_map,
    to_results_rows,
)

XES_PATH = REPO_ROOT / "evaluation" / "Road_Traffic_Fine_Management_Process.xes"
MODEL_PATH = REPO_ROOT / "evaluation" / "rtfm.yaml"
DATA_DIR = REPO_ROOT / "evaluation" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_CACHE = DATA_DIR / "rtfm_event_log.parquet"
RESULTS_CSV = DATA_DIR / "rtfm_conformance_results.csv"

TRIGGER_ACTIVITY = "Create Fine"

# Number of cases to check. Set to None to run the full log (slow: the checker
# loops cases in Python).
SAMPLE_N = 5000
RANDOM_STATE = 42

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

/mnt/e/Uni/Master/thesis/code/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the XES log

Parsing 185&nbsp;MB of XES takes several minutes, so the normalised frame is cached as parquet and
reused on later runs. Delete `evaluation/data/rtfm_event_log.parquet` to force a re-parse.

Normalisation is deliberately minimal: rename the three XES standard columns to the canonical
`case_id` / `activity` / `timestamp`, drop bookkeeping columns, make timestamps tz-naive UTC
(episode arithmetic compares bare timestamps), and **keep every payload attribute as its own
column** so data conditions can address them.

In [2]:
ATTRIBUTE_COLUMNS = [
    "amount",
    "expense",
    "paymentAmount",
    "totalPaymentAmount",
    "points",
    "article",
    "vehicleClass",
    "dismissal",
    "notificationType",
    "lastSent",
]


def load_rtfm_log(force_reparse: bool = False) -> pd.DataFrame:
    """Return the RTFM log as a wide event-log DataFrame, using the parquet cache."""
    if PARQUET_CACHE.exists() and not force_reparse:
        return pd.read_parquet(PARQUET_CACHE)

    import pm4py

    raw = pm4py.read_xes(str(XES_PATH))
    df = raw.rename(
        columns={
            "case:concept:name": "case_id",
            "concept:name": "activity",
            "time:timestamp": "timestamp",
        }
    )
    keep = ["case_id", "activity", "timestamp"] + [c for c in ATTRIBUTE_COLUMNS if c in df.columns]
    df = df[keep].copy()

    df["case_id"] = df["case_id"].astype(str)
    df["activity"] = df["activity"].astype(str)
    ts = pd.to_datetime(df["timestamp"], utc=True)
    df["timestamp"] = ts.dt.tz_localize(None)

    df = df.sort_values(["case_id", "timestamp"], kind="mergesort").reset_index(drop=True)
    df.to_parquet(PARQUET_CACHE, index=False)
    return df


log = load_rtfm_log()
print(f"{len(log):,} events / {log['case_id'].nunique():,} cases")
print(f"columns: {list(log.columns)}")
assert "itemid" not in log.columns, "an itemid column would disable the checker's wide-log path"
log.head(8)

561,470 events / 150,370 cases
columns: ['case_id', 'activity', 'timestamp', 'amount', 'expense', 'paymentAmount', 'totalPaymentAmount', 'points', 'article', 'vehicleClass', 'dismissal', 'notificationType', 'lastSent']


,case_id,activity,timestamp,amount,expense,paymentAmount,totalPaymentAmount,points,article,vehicleClass,dismissal,notificationType,lastSent
0,A1,Create Fine,2006-07-23 22:00:00,35.0,NaN,NaN,0.0,0.0,157.0,A,NIL,NaN,NaN
1,A1,Send Fine,2006-12-04 23:00:00,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A100,Create Fine,2006-08-01 22:00:00,35.0,NaN,NaN,0.0,0.0,157.0,A,NIL,NaN,NaN
3,A100,Send Fine,2006-12-11 23:00:00,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A100,Insert Fine Notification,2007-01-14 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P,P
5,A100,Add penalty,2007-03-15 23:00:00,71.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,A100,Send for Credit Collection,2009-03-29 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,A10000,Create Fine,2007-03-08 23:00:00,36.0,NaN,NaN,0.0,0.0,157.0,A,NIL,NaN,NaN


In [3]:
log["activity"].value_counts().to_frame("events")

,events
activity,
Create Fine,150370
Send Fine,103987
Insert Fine Notification,79860
Add penalty,79860
Payment,77601
Send for Credit Collection,59013
Insert Date Appeal to Prefecture,4188
Send Appeal to Prefecture,4141
Receive Result Appeal from Prefecture,999


## 2. Outcome preprocessing

Each episode gets a binary outcome label, needed by the statistical tests. An episode is **good**
when the full amount owed was paid *in time*; the process signals "in time" through its own
escalation events rather than an assumed statutory deadline:

```
due  = amount at 'Create Fine'  +  sum of 'expense' in the episode
paid = max(totalPaymentAmount)  (cumulative attribute)  else sum(paymentAmount)

good = paid >= due
       AND (no 'Add penalty', or the last payment preceded it)
       AND no 'Send for Credit Collection'
```

Two attribute quirks drive the arithmetic (see `derive_episode_outcome` in `rtfm_utils.py`):
`Add penalty` **rewrites** `amount` to the penalised total, so the amount owed is read from the
`Create Fine` event only; and `totalPaymentAmount` is **cumulative**, so it is maxed, never summed.

The labels are the literal strings `"Success"` and `"Failure"` because
`derive/statistical_analysis.py` substring-matches those words when binarising.

In [4]:
# Hand-checks against the raw XES before running at scale.
#   A1   -- fine 35.0 + expense 11.0, never paid                -> Failure
#   A100 -- penalty added, sent for credit collection           -> Failure
for case_id in ["A1", "A100"]:
    trace = log[log["case_id"] == case_id]
    label, details = derive_episode_outcome(trace)
    print(f"{case_id}: {label}")
    print(f"   activities: {list(trace['activity'])}")
    print(f"   {details}")

# A case paid on the spot, for the positive side of the check.
paid_cases = set(log.loc[log["activity"] == "Payment", "case_id"])
short_paid = [c for c in list(paid_cases)[:2000] if len(log[log["case_id"] == c]) == 2]
if short_paid:
    case_id = short_paid[0]
    trace = log[log["case_id"] == case_id]
    label, details = derive_episode_outcome(trace)
    print(f"\n{case_id}: {label}")
    print(f"   activities: {list(trace['activity'])}")
    print(f"   {details}")

A1: Failure
   activities: ['Create Fine', 'Send Fine']
   {'amount_due': 46.0, 'amount_paid': 0.0, 'paid_in_full': False, 'paid_before_penalty': True, 'credit_collection': False, 'penalty_ts': None, 'last_payment_ts': None}
A100: Failure
   activities: ['Create Fine', 'Send Fine', 'Insert Fine Notification', 'Add penalty', 'Send for Credit Collection']
   {'amount_due': 46.0, 'amount_paid': 0.0, 'paid_in_full': False, 'paid_before_penalty': False, 'credit_collection': True, 'penalty_ts': Timestamp('2007-03-15 23:00:00'), 'last_payment_ts': None}

S98220: Success
   activities: ['Create Fine', 'Payment']
   {'amount_due': 33.6, 'amount_paid': 33.6, 'paid_in_full': True, 'paid_before_penalty': True, 'credit_collection': False, 'penalty_ts': None, 'last_payment_ts': Timestamp('2004-09-07 22:00:00')}


In [5]:
all_case_ids = log["case_id"].drop_duplicates()
if SAMPLE_N is not None and SAMPLE_N < len(all_case_ids):
    case_ids = set(all_case_ids.sample(n=SAMPLE_N, random_state=RANDOM_STATE))
else:
    case_ids = set(all_case_ids)

sample = log[log["case_id"].isin(case_ids)].copy()
print(f"working set: {sample['case_id'].nunique():,} cases / {len(sample):,} events")

# Outcome balance over whole cases (episode-level balance is printed after segmentation).
preview = (
    sample.groupby("case_id", sort=False)
    .apply(lambda t: derive_episode_outcome(t)[0], include_groups=False)
    .value_counts()
)
print(preview)
print(f"\ngood share: {preview.get('Success', 0) / preview.sum():.1%}")

working set: 5,000 cases / 18,749 events
Failure    3371
Success    1629
Name: count, dtype: int64

good share: 32.6%


## 3. The Declare model

`evaluation/rtfm.yaml` is loaded by the same `load_model_from_yaml` the SBT pipeline uses. Edit that
file to add or change rules — every template in `conformance.types.DeclareTemplate` is available,
along with `time_condition`, `activation_condition` / `correlation_condition` (data-aware, addressing
event attributes as `items['amount']`), `target_selection`, and the context windows.

The non-schema `severity` key (`mandatory` | `informational`) decides which pair of result columns a
constraint lands in; both are analysed.

In [6]:
model = load_model_from_yaml(MODEL_PATH)
severity_map = load_severity_map(MODEL_PATH)

pd.DataFrame(
    [
        {
            "id": c.id,
            "template": c.template.value,
            "activation": c.activation,
            "target": c.target,
            "condition": c.activation_cond.expression if c.activation_cond else "",
            "time_window": (
                f"[{c.time_condition.min_seconds}, {c.time_condition.max_seconds}] s"
                if c.time_condition
                else ""
            ),
            "severity": severity_map[c.id],
            "label": c.label,
        }
        for c in model.constraints
    ]
)

,id,template,activation,target,condition,time_window,severity,label
0,RTF-01,Init,Create Fine,NaN,,,mandatory,Every case starts with fine creation
1,RTF-02,Precedence,Send Fine,Create Fine,,,mandatory,A fine must be created before it is sent
2,RTF-03,Response,Send Fine,Insert Fine Notification,,"[0.0, 7776000.0] s",mandatory,A sent fine must be notified within 90 days
3,RTF-04,Precedence,Add penalty,Insert Fine Notification,,,mandatory,Penalty may only be added after notification
4,RTF-05,Precedence,Send for Credit Collection,Add penalty,,,mandatory,Credit collection only after a penalty was added
5,RTF-06,At Most One,Add penalty,NaN,,,informational,Penalty is added at most once
6,RTF-07,Response,Insert Date Appeal to Prefecture,Send Appeal to Prefecture,,,mandatory,A prefecture appeal must be sent after it is r...
7,RTF-08,Response,Receive Result Appeal from Prefecture,Notify Result Appeal to Offender,,,mandatory,A prefecture result must be notified to the of...
8,RTF-09,Response,Create Fine,Send Fine,(items['amount'] > 50),,informational,Large fines must be sent to the offender
9,RTF-10,Not Co-Existence,Send for Credit Collection,Payment,,,informational,A case sent for credit collection is not also ...


## 4. Episode segmentation and conformance checking

Each case is segmented by `build_trigger_episodes` — the generalised copy of
`main_per_sbt._build_trigger_episodes`, with `Create Fine` as the trigger instead of
`InvasiveVentilationStarted`. An episode runs from its trigger to the earliest of the next trigger
and the end of the trace. RTFM cases carry exactly one `Create Fine`, so in practice this yields one
episode per fine; the segmentation is kept because it is the pipeline's episode definition, and it
generalises if a repeated trigger is chosen instead.

Constraints are then evaluated per episode with `evaluate_constraint`, the same call
`main_per_sbt._process_stay` makes.

In [7]:
def check_case(case_id: str, trace: pd.DataFrame) -> list[dict]:
    """Segment one case into episodes and evaluate every constraint on each."""
    trace = trace.sort_values("timestamp")
    windows = build_trigger_episodes(trace, trigger_activity=TRIGGER_ACTIVITY)

    episodes = []
    for start_ts, end_ts in windows:
        episode = trace[(trace["timestamp"] >= start_ts) & (trace["timestamp"] < end_ts)]
        if episode.empty:
            continue
        label, details = derive_episode_outcome(episode)
        results = [
            evaluate_constraint(case_id=case_id, trace_df=episode, constraint=c)
            for c in model.constraints
        ]
        episodes.append(
            {
                "start": start_ts,
                "end": end_ts,
                "outcome": label,
                "details": details,
                "results": results,
            }
        )
    return to_results_rows(case_id, episodes, severity_map)


rows = []
for case_id, trace in tqdm(sample.groupby("case_id", sort=False), total=sample["case_id"].nunique()):
    rows.extend(check_case(str(case_id), trace))

results = pd.DataFrame(rows)
results.to_csv(RESULTS_CSV, index=False)
print(f"{len(results):,} episodes -> {RESULTS_CSV}")
results.head()

100%|██████████| 5000/5000 [03:12<00:00, 26.03it/s]


5,000 episodes -> /mnt/e/Uni/Master/thesis/code/evaluation/data/rtfm_conformance_results.csv


,stay_id,episode_index,episode_start,episode_end,sbt_attempt,sbt_outcome,next_sbt_outcome,mandatory_violated_constraints,mandatory_satisfied_constraints,mandatory_vacuous_constraints,info_violated_constraints,info_satisfied_constraints,info_vacuous_constraints,amount_due,amount_paid,paid_in_full,paid_before_penalty,credit_collection
0,A10049,0,2007-03-27 22:00:00,2007-03-29 22:00:01,1,Success,,,RTF-01,RTF-02;RTF-03;RTF-04;RTF-05;RTF-07;RTF-08,,RTF-06;RTF-10,RTF-09,36.0,36.0,True,True,False
1,A10130,0,2007-03-09 23:00:00,2008-10-30 23:00:01,1,Failure,,,RTF-01;RTF-02;RTF-03;RTF-04,RTF-05;RTF-07;RTF-08,,RTF-06;RTF-10,RTF-09,49.0,87.0,True,False,False
2,A10161,0,2007-03-07 23:00:00,2008-10-14 22:00:01,1,Failure,,,RTF-01;RTF-02;RTF-03;RTF-04,RTF-05;RTF-07;RTF-08,,RTF-06;RTF-10,RTF-09,49.0,87.0,True,False,False
3,A10171,0,2007-03-09 23:00:00,2009-03-29 22:00:01,1,Failure,,,RTF-01;RTF-02;RTF-03;RTF-04;RTF-05,RTF-07;RTF-08,,RTF-06;RTF-10,RTF-09,49.0,0.0,False,False,True
4,A10187,0,2007-03-11 23:00:00,2007-08-19 22:00:01,1,Failure,,RTF-03,RTF-01;RTF-02,RTF-04;RTF-05;RTF-07;RTF-08,,RTF-06;RTF-10,RTF-09,54.6,0.0,False,True,False


### Sanity checks

Before reading anything into the statistics, confirm the model actually engaged with the log:

* **RTF-01** (`INIT Create Fine`) must be satisfied for 100% of episodes — every case starts there,
  so anything else means segmentation or slicing is wrong.
* **RTF-09** must be non-vacuous somewhere — that is the proof the wide-log path resolves
  `items['amount']`.
* No rule may be vacuous for 100% of episodes — that would mean an activity name doesn't match.

In [8]:
def rule_state_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Per-rule counts of satisfied / violated / vacuous episodes."""
    counts = {}
    for state in ("satisfied", "violated", "vacuous"):
        exploded = (
            pd.concat(
                [df[f"mandatory_{state}_constraints"], df[f"info_{state}_constraints"]]
            )
            .fillna("")
            .astype(str)
            .str.split(";")
            .explode()
        )
        counts[state] = exploded[exploded != ""].value_counts()
    out = pd.DataFrame(counts).fillna(0).astype(int).sort_index()
    out["n"] = out.sum(axis=1)
    out["violation_rate"] = (out["violated"] / (out["satisfied"] + out["violated"])).round(4)
    return out


states = rule_state_counts(results)
display(states)

n_episodes = len(results)
assert states.loc["RTF-01", "satisfied"] == n_episodes, "RTF-01 must hold for every episode"
assert states.loc["RTF-09", "vacuous"] < n_episodes, "RTF-09 never activated: items['amount'] unresolved"
always_vacuous = states.index[states["vacuous"] == n_episodes].tolist()
assert not always_vacuous, f"never activated (check activity names): {always_vacuous}"
print("sanity checks passed")

print("\nepisode outcome balance:")
print(results["sbt_outcome"].value_counts())

,satisfied,violated,vacuous,n,violation_rate
RTF-01,5000,0,0,5000,0.0000
RTF-02,3406,74,1520,5000,0.0213
RTF-03,2583,897,1520,5000,0.2578
RTF-04,2669,0,2331,5000,0.0000
RTF-05,1955,0,3045,5000,0.0000
RTF-06,5000,0,0,5000,0.0000
RTF-07,159,1,4840,5000,0.0062
RTF-08,33,4,4963,5000,0.1081
RTF-09,529,133,4338,5000,0.2009
RTF-10,4924,76,0,5000,0.0152


sanity checks passed

episode outcome balance:
sbt_outcome
Failure    3371
Success    1629
Name: count, dtype: int64


### Caveat: date-granular timestamps

Every RTFM event is stamped at local midnight, so the log has **day resolution and no intra-day
order**. Ordering templates compare timestamps strictly, so two activities recorded on the same day
tie and count as *not* ordered. Concretely, 2,894 of 103,987 cases record `Create Fine` and
`Send Fine` on the same date; those come out as RTF-02 violations even though the fine plainly was
created first.

This is a property of the dataset, not of the checker, and it is worth stating when reporting
violation rates for any ordering rule (RTF-02, RTF-04, RTF-05). If same-day pairs should count as
ordered, give the rule a `time_condition` with a negative `min_hours` (e.g. `min_hours: -24`) rather
than nudging the timestamps.

## 5. Statistical analysis

`derive/statistical_analysis.py` is used unchanged. For each rule it builds a 2×2 table of
*violated vs. satisfied* against *bad vs. good outcome*, then reports Fisher's exact test with
Benjamini–Hochberg correction, odds ratio and risk ratio with confidence intervals.

Two vacuity modes are run, mirroring `statistical_analysis.__main__`:

* **activations-only** (`pool_vacuous=False`) — episodes where the rule never activated are excluded.
* **pooled** (`pool_vacuous=True`) — vacuous episodes count as satisfied.

`inter_episode_rules=set()`: no rule here declares `outcome_scope: next_episode`, and with one
episode per fine there is no successor to reference. `load_inter_episode_rules()` is deliberately not
called — its default path points at the SBT config.

In [9]:
from derive.statistical_analysis import run_analysis

df_results = pd.read_csv(RESULTS_CSV)

summaries = {}
for pool_vacuous, tag in [(False, "activations_only"), (True, "pooled")]:
    summaries[tag] = run_analysis(
        df_results,
        pool_vacuous=pool_vacuous,
        inter_episode_rules=set(),
        out_table=DATA_DIR / f"rtfm_case_table_{tag}.csv",
        out_summary=DATA_DIR / f"rtfm_stats_{tag}.csv",
    )

[activations-only] analysis table with 5000 SBT cases.
  table   -> /mnt/e/Uni/Master/thesis/code/evaluation/data/rtfm_case_table_activations_only.csv
  summary -> /mnt/e/Uni/Master/thesis/code/evaluation/data/rtfm_stats_activations_only.csv
[pooled] analysis table with 5000 SBT cases.
  table   -> /mnt/e/Uni/Master/thesis/code/evaluation/data/rtfm_case_table_pooled.csv
  summary -> /mnt/e/Uni/Master/thesis/code/evaluation/data/rtfm_stats_pooled.csv


In [10]:
summary = summaries["activations_only"].copy()
summary["label"] = summary["rule_name"].map({c.id: c.label for c in model.constraints})
summary = summary.sort_values("p_raw")
summary[
    [
        "rule_name",
        "label",
        "n_exposed",
        "n_unexposed",
        "a",
        "b",
        "c",
        "d",
        "p_raw",
        "or",
        "or_ci_low",
        "or_ci_high",
        "rr",
        "rr_ci_low",
        "rr_ci_high",
    ]
]

,rule_name,label,n_exposed,n_unexposed,a,b,c,d,p_raw,or,or_ci_low,or_ci_high,rr,rr_ci_low,rr_ci_high
8,RTF-09,Large fines must be sent to the offender,133,529,97,36,18,511,2.253484e-66,76.492284,41.728067,140.219039,21.434002,13.453975,34.147263
2,RTF-03,A sent fine must be notified within 90 days,897,2583,109,788,1,2582,3.306181e-65,357.154822,49.778537,2562.541479,313.876254,43.881565,2245.095499
9,RTF-10,A case sent for credit collection is not also ...,76,4924,0,76,1629,3295,1.008439e-13,0.013218,0.000819,0.213393,0.019626,0.001238,0.311059
1,RTF-02,A fine must be created before it is sent,74,3406,0,74,110,3296,1.734843e-01,0.200219,0.012327,3.251939,0.205551,0.012897,3.275995
4,RTF-05,Credit collection only after a penalty was added,0,1955,0,0,0,1955,1.000000e+00,3911.000000,32.147796,475799.992043,1956.000000,65.636146,58290.077211
5,RTF-06,Penalty is added at most once,0,5000,0,0,1629,3371,1.000000e+00,2.069040,0.041034,104.327036,1.534520,0.216062,10.898475
0,RTF-01,Every case starts with fine creation,0,5000,0,0,1629,3371,1.000000e+00,2.069040,0.041034,104.327036,1.534520,0.216062,10.898475
3,RTF-04,Penalty may only be added after notification,0,2669,0,0,1,2668,1.000000e+00,1779.000000,25.778808,122769.096981,890.000000,70.891934,11173.344411
7,RTF-08,A prefecture result must be notified to the of...,4,33,0,4,0,33,1.000000e+00,7.444444,0.130790,423.730170,6.800000,0.151222,305.775482
6,RTF-07,A prefecture appeal must be sent after it is r...,1,159,0,1,1,158,1.000000e+00,35.222222,0.980040,1265.871615,26.666667,1.495614,475.464445


In [11]:
# Side-by-side: does pooling vacuous episodes change the direction of any effect?
comparison = (
    summaries["activations_only"][["rule_name", "n_exposed", "or", "p_raw"]]
    .merge(
        summaries["pooled"][["rule_name", "n_exposed", "or", "p_raw"]],
        on="rule_name",
        suffixes=("_activations", "_pooled"),
    )
    .sort_values("rule_name")
)
comparison

,rule_name,n_exposed_activations,or_activations,p_raw_activations,n_exposed_pooled,or_pooled,p_raw_pooled
0,RTF-01,0,2.069040,1.000000e+00,0,2.069040,1.000000e+00
1,RTF-02,74,0.200219,1.734843e-01,74,0.013581,3.148280e-13
2,RTF-03,897,357.154822,3.306181e-65,897,0.235061,1.831039e-53
3,RTF-04,0,1779.000000,1.000000e+00,0,2.069040,1.000000e+00
4,RTF-05,0,3911.000000,1.000000e+00,0,2.069040,1.000000e+00
5,RTF-06,0,2.069040,1.000000e+00,0,2.069040,1.000000e+00
6,RTF-07,1,35.222222,1.000000e+00,1,0.689475,1.000000e+00
7,RTF-08,4,7.444444,1.000000e+00,4,0.229621,3.109323e-01
8,RTF-09,133,76.492284,2.253484e-66,133,5.865517,5.225689e-22
9,RTF-10,76,0.013218,1.008439e-13,76,0.013218,1.008439e-13


## 6. Optional: weighted conformance scoring

Not run here. To add Sugeno/fuzzy weighted trace scores, write an
`evaluation/rtfm_weights.yaml` in the format `conformance/weighted_scoring.py` expects
(`weights: {RTF-01: {metadata_scores: {...}, theta: ..., r_acc: ...}, ...}`), then:

```python
from conformance.weighted_scoring import load_constraints_with_weights, score_trace_collection
constraints = load_constraints_with_weights(MODEL_PATH, DATA_DIR.parent / "rtfm_weights.yaml")
scores_df, detail_df, kpi = score_trace_collection(trace_results, constraints)
```

That needs `TraceResult` objects rather than the flattened rows written above — collect them in
`check_case` if this is wanted.

## Running the full log

Set `SAMPLE_N = None` and re-run from section 2. The checker loops cases in Python, so all 150k cases
take a while; if that becomes the bottleneck, chunk `case_ids` and map `check_case` over a
`concurrent.futures.ProcessPoolExecutor`, the way `main_per_sbt.py` parallelises over stays.